In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

# Text Mining
import nltk
nltk.download('punkt')
from wordcloud import WordCloud
from collections import Counter
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt_tab')

# warning management library
import warnings

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\smazo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
warnings.filterwarnings('ignore')
sns.set()
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

### 1. Loading of MSPT dataset and imbd titles

In [3]:
mpst = pd.read_csv("../Raw_Data/mpst_full_data.csv")
# https://www.kaggle.com/datasets/cryptexcode/mpst-movie-plot-synopses-with-tags?resource=download

In [4]:
mpst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14828 entries, 0 to 14827
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   imdb_id          14828 non-null  object
 1   title            14828 non-null  object
 2   plot_synopsis    14828 non-null  object
 3   tags             14828 non-null  object
 4   split            14828 non-null  object
 5   synopsis_source  14828 non-null  object
dtypes: object(6)
memory usage: 695.2+ KB


In [5]:
mpst.head()

,imdb_id,title,plot_synopsis,tags,split,synopsis_source
0,tt0057603,I tre volti della paura,Note: this synopsis is for the orginal Italian...,"cult, horror, gothic, murder, atmospheric",train,imdb
1,tt1733125,Dungeons & Dragons: The Book of Vile Darkness,"Two thousand years ago, Nhagruul the Foul, a s...",violence,train,imdb
2,tt0033045,The Shop Around the Corner,"Matuschek's, a gift store in Budapest, is the ...",romantic,test,imdb
3,tt0113862,Mr. Holland's Opus,"Glenn Holland, not a morning person by anyone'...","inspiring, romantic, stupid, feel-good",train,imdb
4,tt0086250,Scarface,"In May 1980, a Cuban man named Tony Montana (A...","cruelty, murder, dramatic, cult, violence, atm...",val,imdb


In [6]:
mpst.iloc[0,2]

'Note: this synopsis is for the orginal Italian release with the segments in this certain order.Boris Karloff introduces three horror tales of the macabre and the supernatural known as the \'Three Faces of Fear\'.THE TELEPHONERosy (Michele Mercier) is an attractive, high-priced Parisian call-girl who returns to her spacious, basement apartment after an evening out when she immediately gets beset by a series of strange phone calls. The caller soon identified himself as Frank, her ex-pimp who has recently escaped from prison. Rosy is terrified for it was her testimony that landed the man in jail. Looking for solace, Rosy phones her lesbian lover Mary (Lynda Alfonsi). The two women have been estranged for some time, but Rosy is certain that she is the only one who can help her. Mary agrees to come over that night. Seconds later, Frank calls again, promising that no matter who she calls for protection, he will have his revenge. Unknown to Rosy, Mary is the caller impersonating Frank. Marry

In [7]:
mpst_short = mpst[["imdb_id","plot_synopsis", "tags"]]

In [8]:
titles = pd.read_csv("../Raw_Data/title.basics.tsv", sep='\t') #, nrows = 10000)
#https://developer.imdb.com/non-commercial-datasets/

In [9]:
titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12536830 entries, 0 to 12536829
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          object
 1   titleType       object
 2   primaryTitle    object
 3   originalTitle   object
 4   isAdult         int64 
 5   startYear       object
 6   endYear         object
 7   runtimeMinutes  object
 8   genres          object
dtypes: int64(1), object(8)
memory usage: 860.8+ MB


In [10]:
titles.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [11]:
titles_short = titles[["tconst", "primaryTitle", "genres"]]

In [12]:
movies = titles_short.merge(mpst_short, left_on="tconst", right_on="imdb_id")
movies.head()

,tconst,primaryTitle,genres,imdb_id,plot_synopsis,tags
0,tt0000091,The House of the Devil,"Horror,Short",tt0000091,The film opens with a large bat flying into a ...,"paranormal, gothic"
1,tt0000225,Beauty and the Beast,"Family,Fantasy,Romance",tt0000225,A widower merchant lives in a mansion with his...,fantasy
2,tt0000230,Cinderella,"Drama,Family,Fantasy",tt0000230,"A prologue in front of the curtain, suppressed...",fantasy
3,tt0000417,A Trip to the Moon,"Adventure,Comedy,Fantasy",tt0000417,"At a meeting of the Astronomic Club, its presi...","psychedelic, satire"
4,tt0000488,The Land Beyond the Sunset,"Drama,Fantasy,Short",tt0000488,Joe is an impoverished New York newsboy who li...,"fantasy, storytelling"


In [13]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14807 entries, 0 to 14806
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tconst         14807 non-null  object
 1   primaryTitle   14807 non-null  object
 2   genres         14807 non-null  object
 3   imdb_id        14807 non-null  object
 4   plot_synopsis  14807 non-null  object
 5   tags           14807 non-null  object
dtypes: object(6)
memory usage: 694.2+ KB


In [15]:
movies["genres"] = movies["genres"].apply(lambda x: x.replace(",", " "))
movies["tags"] = movies["tags"].apply(lambda x: x.replace(",", ""))

In [16]:
movies.to_csv("../Transformed_Data/movies_info.csv", index=None)

In [17]:
movies.head()

,tconst,primaryTitle,genres,imdb_id,plot_synopsis,tags
0,tt0000091,The House of the Devil,Horror Short,tt0000091,The film opens with a large bat flying into a ...,paranormal gothic
1,tt0000225,Beauty and the Beast,Family Fantasy Romance,tt0000225,A widower merchant lives in a mansion with his...,fantasy
2,tt0000230,Cinderella,Drama Family Fantasy,tt0000230,"A prologue in front of the curtain, suppressed...",fantasy
3,tt0000417,A Trip to the Moon,Adventure Comedy Fantasy,tt0000417,"At a meeting of the Astronomic Club, its presi...",psychedelic satire
4,tt0000488,The Land Beyond the Sunset,Drama Fantasy Short,tt0000488,Joe is an impoverished New York newsboy who li...,fantasy storytelling
